# Introduction
Process mining helps uncover insights from data that represent different events in a process. It is valuable for understanding workflows, finding inefficiencies, and optimizing development processes. Using Kaiaulu  for process mining enables this discovery for Github Events (e.g. creating issues, push events, deleting branches, etc). Allowing users to visualize and analyze event data effectively. This notebook demonstrates how to use Kaiaulu to download and parse Github Event data. As well how to interact with the data to leverage process discovery for deeper analysis. This [video](https://www.youtube.com/watch?v=XLHtvt36g6U&t=1181s) provides an overview of process mining concepts and its applications and can be used as a reference point to begin. This notebook heavily relies on the python package [pm4py](https://github.com/process-intelligence-solutions/pm4py) to create event mappings.

This notebook walks through multiple different models and algorithms for modeling the process. It is reccomended you start with a small number of event issues and gradually build multiple versions to have a deeper understanding of what events are changed the process model. At the bottom of the notebook is an examples section that shows what the models can generate. 

### Requirements

-   Gihub Access Token 
-   [Kaiaulu](https://github.com/sailuh/kaiaulu) R package to download data via CLI
-   Python environment with [pm4py](https://github.com/process-intelligence-solutions/pm4py) installed
-   Faker installed for CSV data generation


### Python Imports

In [1]:
import pandas
import os
import pm4py
import subprocess

from api.csv_generator import *
from api.process_visual_generation import *

### Download and Parse data with Kaiaulu ghevents.R (CLI)

#### Set Up and Execute Download Command


In [ ]:
# Save and set the current working directory to Kaiaulu.
cwd = os.getcwd()
os.chdir(os.path.expanduser("~/Desktop/Kaiaulu/Working_issues/kaiaulu/"))

# To download use the download command specifing the <config_file> <github_token>.

command = f"Rscript exec/ghevents.R download conf/kaiaulu.yml --token_path=~/.ssh/github_token"
subprocess.run(command, shell=True, check=True)


#### Execute Parse Command

As stated above it is reccomended you start with only a few event issues. To do this you can open the created issue_output.csv with Excel or Google Sheets and modify the table to only include a few. 

Note: commit_output.csv has been implemented for furture development and is note currently used for any process modeling. 

In [ ]:
# To Parse use the parse command and specify the <config_path> <output_dir>.

command = f"Rscript exec/ghevents.R parse conf/kaiaulu.yml ../rawdata/issue_output.csv ../rawdata/commit_output.csv"
subprocess.run(command, shell=True, check=True)

### Generate CSV data

To start using process mining it is recommnend you generate a "fake" .csv event log to see how events change process maps. This is setup below, just generate the csv with the seed set to 2 to start. This function can be used to further explore how different events and number of issues change the graphs generated.

In [2]:
# Example usage 
# Note: Set the output path
output_csv = "../created_csv.csv"
generate_csv_file(num_issues=1, num_events_per_issue=7, output_csv=output_csv, seed=2)

Fake event log data saved to ../created_csv.csv


'../created_csv.csv'

To compare graphs generated we can modify the generated csv with the following function. So there will be two .csv files the original and a modified one with one event changed. 

In [3]:
# Example usage
# Note: Set the output_csv
modify_event_in_csv("../created_csv.csv", row_index=5, new_event="assigned", output_csv="../modified_csv.csv")

Modified CSV saved to ../modified_csv.csv


### Get Start and End Activies From Event Logs

The start and end activies are the first and last recorded events in the process workflow. Marking the entry and exit points where no prior or further events occur. These activies are useful for reasoning through process grpahs and finding inefficencies in a non-graphical manner.

In [ ]:
# Specify the csv file path <csv_path>.
csv_path = "../rawdata/issue_output.csv"

# Call start_end_activities.
start_end_activities(csv_path)

### Generate Tree from Event Logs

The process tree generated from event logs is our first visual we will generate. It uses the Inductive Miner Algorithm to generate a hierarchical visualization of the workflow. Capturing the sequences and dependecies of events. The algorithm is based on process discovery, and will highligh decision points and parallel activities. It is important for understanding the overall flow and structure of events and identifiying transitions and potential ineffcientices in the process. 

We will need a directory to save the images for methods from here on. Specify the output_dir if it does not already exist it will create it.  

In [9]:
# Specify the directory to save images.
output_dir = "../rawdata/images/"

output_dir = os.path.expanduser(output_dir)
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Call generate_tree.
generate_tree_inductive(csv_path, output_dir)

### Generate Graph with or without filtering

We will create a BPMN (Buiness Process Model and Notation) with the same inductive miner from the last part creating the tree.
The BPMN model is a standard graphical representation of the process, it is the simpilest graph we will generate. It highlights the sequences of activies, decision points, and possible parallel paths in the process model. Compared to the tree it is easier to identify ineffcienceies and deviation the process flow from the simplistic nature. 

Filtering can also be applied to the process. By applying filtering we refine the model to focus on the most signifcant part of the process. It does this by reducing the noise and making the key patterns more apparent. This noise could be events that are not required to complete the process or that stray away from the main process for example. You can set the noise threshold as a parameter, note the default is 0.0. 

In [ ]:
# Call generate_graph_filtered and specify a noise threshold. 
generate_graph_inductive(csv_path, output_dir, 0.5)

### Generate Performance Graph: Time or Occurrence Edge Weights

### Time Edge Weight Graph
The time edge weight graph visualizes the process flow with a focus on the time spent between activities. It uses the Performance DFG (Directly-Follows Graph). Every edge has a value which represents the interval between consecutive events. These time-based relations are useful to show where the process may need to be optimized to reduce the overall cycle time in the process. 

In [ ]:
# Call generate_performance_graph.
generate_performance_graph_dfg(csv_path, output_dir)

### Occurrence Edge Weight Graph

The DFG (Directly-Follows Graph) generates the graph with occurrence edges. The weight is based on the frequency of transitions between nodes. This model highlights the most common paths, allowing for a better understanding of the domnant workflow patterns. It can help identify redudent steps in the process. 

In [ ]:
# Call generate_count_graph.
generate_count_graph_dfg(csv_path, output_dir)

### Petri Net

A Petri Net is a commonly used formal model used to represent workflows in automation and process mining. Unlike simplier models such as BPMN, Petri Nets represent conditions, tokens, and transitions. Instead of just showing the overview of the process. 

Important Symbols: 

- Circle with black center dot: Marks the start of the process.
- Circle with black square: Marks the end of the process.
- Empty circles: State or conditions in the process.
- Black boxes: Transitions that may be considered special. Example: Silent steps that don't correspond to events but may be needed for logical execution of the process.

In [ ]:
# Call generate_petri_net.
generate_petri_net_inductive(csv_path, output_dir)

## Examples

To show the functionality of each model we will make it easy by working with only one issue with multiple events. We will use the following issue 2 from Kaiaulu:

![Starting Data](images/starting_data.png)

### Process Tree

![Process Tree](images/process_tree.png)

We see we have generated a simple process tree with sequences and xor actions.

### Process Graph (BPMN)

![Process Graph](images/process_graph.png)

The process graph shows up the simpliest visual. We can start at the green circle. From there the issue was labeld and then was either assigned-milestoned-closed. Or it was unlabled aftering being labeled then referenced-closed. From the closed state the process either ended or the issue was reopened and repeated.

### Performance Graph (Time Weighted)

![Time Weighted Graph](images/performance_dfg.png)

The time weighted graph starts with the circle with the center dot. From there it goes through the process above but each edge has a weight based on the total time between events. We can see the time from the issue being referenced-closed was 9 months. Suggesting maybe someone forgot to close the event. This indicates a inefficiency within this Github issue process.

### Performance Graph (Occurrence Weighted)

![Occurrence Weighted](images/occurrence_dfg.png)

With the occurrence weighted graph we see different colors for nodes annotated with multiple occurrences. To be exact the "closed" and "labled" event happen twice. Showing us the important points in the issue process.

### Petri Net Graph

![Petri Net](images/petri_net.png)

Finally, the Petri Net is similar to the process graph. Except we can see states in the process represented by circles. As well as the special transitions represented by the black boxes.







